In [2]:
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownTextSplitter
from pathlib import Path
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
import os
from langchain_chroma import Chroma
import glob

load_dotenv()

loader = PDFPlumberLoader(str(Path("../../docs/Mani-Resume.pdf").resolve()))
data = loader.load()
print("data", [page.page_content for page in data])


c:\Users\dell\Desktop\Mani\Langchain\Learning\langraph-practice-app\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data ['MANI KUMAR POPURI\nSENIOR FRONT END DEVELOPER\nFront end Developer with close to 7 Years of Experience in Web Development and Strong\nProblem-Solving Skills | Adaptable to New Technologies | Specialized in Delivering Complex\nFeatures\nSKILLS Angular (Components, Services, RxJS, Routing, Forms, NgRx) | Rxjs | ReactJS |\nHTML | CSS | Javascript | Redux | GraphQL | RESTAPI\nTypescript | Jest | ExpressJs | NodeJs | NextJs(App router) | Elastic Search\nEXPERIENCE CONSULTANT\nDeloitte | APR 2024 - Present\nJoined Deloitte as a React Developer, delivering scalable, high-\nperformance applications within the Financial Services domain for\nglobal banking-type clients\nBuilt reusable, maintainable UI components using React,\nTypeScript, and modern frontend best practices, ensuring\nresponsiveness and accessibility compliance\nRapidly transitioned from React to Angular, quickly gaining\nproficiency in Angular architecture, components, services, RxJS,\nrouting, NgRx (Store) and forms\nCont

In [3]:
from langchain_text_splitters import MarkdownTextSplitter, RecursiveCharacterTextSplitter

text_splitter = MarkdownTextSplitter(chunk_size=1000, chunk_overlap=200)
recursive_text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200) 
texts = text_splitter.split_documents(data)
texts = [
    type(doc)(page_content=doc.page_content, metadata={**doc.metadata, "chunk_order": i, "chunk_index": i})
    for i, doc in enumerate(texts)
]
for i, text in enumerate(texts):
    print(f"Chunk {i+1}:\n{text.page_content, text.metadata}\n{'-'*40}")


Chunk 1:
('MANI KUMAR POPURI\nSENIOR FRONT END DEVELOPER\nFront end Developer with close to 7 Years of Experience in Web Development and Strong\nProblem-Solving Skills | Adaptable to New Technologies | Specialized in Delivering Complex\nFeatures\nSKILLS Angular (Components, Services, RxJS, Routing, Forms, NgRx) | Rxjs | ReactJS |\nHTML | CSS | Javascript | Redux | GraphQL | RESTAPI\nTypescript | Jest | ExpressJs | NodeJs | NextJs(App router) | Elastic Search\nEXPERIENCE CONSULTANT\nDeloitte | APR 2024 - Present\nJoined Deloitte as a React Developer, delivering scalable, high-\nperformance applications within the Financial Services domain for\nglobal banking-type clients\nBuilt reusable, maintainable UI components using React,\nTypeScript, and modern frontend best practices, ensuring\nresponsiveness and accessibility compliance\nRapidly transitioned from React to Angular, quickly gaining\nproficiency in Angular architecture, components, services, RxJS,\nrouting, NgRx (Store) and forms',

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
DB_NAME = str(Path("./").parent.parent / "db" / "mani_resume_db")
print(str(Path("../../docs/db/mani_resume_db").resolve()))
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

if os.path.exists(DB_NAME):
        Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()
vectorstore = Chroma.from_documents(texts, embeddings, persist_directory=DB_NAME)
collection = vectorstore._collection
count = collection.count()
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimension = len(sample_embedding)
print(f"There are {count} vectors with dimension {dimension} in the vectorstore.")

C:\Users\dell\Desktop\Mani\Langchain\Learning\langraph-practice-app\docs\db\mani_resume_db


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3968.27it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


There are 6 vectors with dimension 768 in the vectorstore.


In [5]:
vector_store=Chroma(persist_directory=DB_NAME, embedding_function=HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2"))
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 10})
question = "what are roles and responsibilities of Mani while working at Publicis Sapient?"
docs = retriever.invoke(question)

context = "\n\n".join(doc.page_content for doc in docs)
print("Context retrieved from vectorstore:\n", context)



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4678.60it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Context retrieved from vectorstore:
 Product Owners to collaborating through the development phase,
coordinating with testers, and collecting client feedback for
enhancements or suggestions.
Proficient in Resolving Critical Production Defects through AWS
Log Analysis and Swift Action | Extensive Experience in Debugging
and Troubleshooting Critical Issues.
Collaborated in Agile/Scrum teams, participating in daily stand-
ups, sprint planning, and code reviews.
Deploying Features after sign-off using Jenkins CI/CD
pipelines.Managing code repos using GIT.
Experienced in conducting client demos to gather feedback and
secure approvals, ensuring alignment with client expectations and
project objectives.

ASSOCIATE SOFTWARE ENGINEER
Publicis Sapient | MAR 2022 - JUL 2023
Developed various Features, Crafted multiple reusable React ES6
components following the atomic design lifecycle, driving code
reusability and reducing tech debt.
Contributed in streamlining a complex booking reservation syste

In [6]:
from langchain.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq


system_prompt = """You are a helpful AI assistant who is reviewing the resume of a job candidate and answers questions based on the  information provided in context below
Use the given context only to answer the question and be descriptive about the role.
If you don't know the answer, say you don't know.
context: {context}
do not list the chunks in your answer. Answer in points if applicable
"""
llm = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=os.getenv("GROQ_API_KEY"), temperature=0.7)
formatted_prompt = system_prompt.format(context=context)
messages = [SystemMessage(content=formatted_prompt), HumanMessage(content=question)]
response = llm.invoke(messages)
print(response.content)

Based on the context provided, here are the roles and responsibilities of Mani while working at Publicis Sapient:

1. **Associate Software Engineer (Mar 2022 - Jul 2023)**:
	* Developed various features.
	* Crafted multiple reusable React ES6 components following the atomic design lifecycle.
	* Driven code reusability and reduced tech debt.
	* Contributed to streamlining a complex booking reservation system into a user-friendly Single Page GDP approach.
	* Improved conversion rates by ~33% and reduced user drop-offs.
	* Skilled in front-end data integration, adept at backend communication, and experienced in creating interactive features that dynamically retrieve data based on user actions using GraphQl schema and resolvers.
	* Experienced in crafting user-friendly, responsive websites for optimal cross-platform experiences.
	* Collaborated effectively with UX designers to grasp requirements and translate them into intuitive UI screens.
	* Proficient in crafting test cases for componen